# Gasificador 0D — Señales de BC: rampas, escalones, pulsos y control

**Objetivo:** Demostrar el uso de señales temporales y de retroalimentación de estado en las
condiciones de contorno del gasificador. Todas las simulaciones son en 0D (N=1, sin gradientes
axiales) sobre el caso base batch/semibatch de pirólisis.

**Señales probadas:**

| Sección | BC modificada | Tipo de señal | Descripción |
|---------|--------------|---------------|-------------|
| 3A | `T_wall` | `ramp(t)` | Calentamiento gradual de pared hasta 800 °C |
| 3B | `T_wall` | `step(t)` | Escalón de temperatura: precalentamiento → operación |
| 3C | `Qwall` | `pulse(t)` | Inyección periódica de calor al reactor |
| 3D | `T_wall` | `piecewise(t)` | Perfil de operación segmentado: calentamiento → plateau → enfriamiento |
| 3E | `T_wall` | `proportional(t,snap)` | Control P: mantener Ts_mean en SP ajustando T_wall |

**Referencia de arquitectura:** `.claude/equipment/signal-bc-integration.md`  
**Módulos de señal:** `src.control.signals`, `src.control.controllers`, `src.utils.signals`

**Condiciones de contorno base comunes a todos los casos:**
- Modo: batch sellado (v_out=0) — sin flujo de gas
- Biomasa: softwood_spruce, mc_wb=16.5 %
- Presión: 1.01325 bar (atmosférica)
- Duración: 3600 s

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.solvers.runner_gasifier                  import run_step
from src.units.gasifier.state_extraction          import build_gasifier_results
from src.postprocessing.gasifier_balances         import check_balances
from src.postprocessing.gasifier_plots            import plot_solid_profiles, plot_gas_profiles
from src.io.fuels_reader                          import read_fueldb
from src.units.gasifier.config.boundary_c         import build_bc_config
from src.units.gasifier.config.thermal_bc         import build_thermal_bc_config
from src.units.gasifier.config.transport          import build_transport_config
from src.units.gasifier.config.gas_props          import build_gas_prop_config, GASIFIER_GAS_SPECIES
from src.units.gasifier.config.solid_props        import build_solid_prop_config
from src.units.gasifier.config.initial_c          import build_initial_c_config
from src.control.signals                          import ramp, step, pulse, piecewise
from src.control.controllers                      import proportional
from src.utils.signals                            import resolve
from src.utils.profiling                          import print_benchmark_functions

FUEL_PATH = os.path.join(ROOT, "materials", "fuels", "softwood_spruce.yaml")
GAS_DB    = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")
SOLID_DB  = os.path.join(ROOT, "materials", "solids", "soliddb.txt")

print("Imports OK")

In [ ]:
# ─── Geometría y parámetros comunes ──────────────────────────────────────────
fuel_config  = read_fueldb(FUEL_PATH)
nc           = 9
N            = 1              # 0D — reactor perfectamente mezclado
species      = list(GASIFIER_GAS_SPECIES)

Di, Do  = 0.10, 0.114         # [m] diámetro interno / externo
e_wall  = (Do - Di) / 2       # [m] espesor de pared (7 mm, acero inox)
L       = 0.50                # [m] longitud del reactor
dz      = L / N               # [m] longitud de celda (= L en 0D)
Ai      = 0.25 * np.pi * Di**2
Pi, Po  = np.pi * Di, np.pi * Do
epsi_r  = 0.60                # [-] porosidad del lecho
dp0     = float(fuel_config["physical"]["dp_initial"])          # [m]
rho_p   = float(fuel_config["physical"]["rho_particle"])        # [kg/m³]
rho_char0 = fuel_config["pyrolysis_yields"]["char"] * rho_p * (1 - epsi_r)

MC_WB   = 0.165               # [-] contenido de humedad en base húmeda
P_OUT   = 1.01325             # [bar]
T_END   = 3600.0              # [s] duración de todos los tests

# Condiciones iniciales comunes
rho_bio_0 = rho_p * (1 - epsi_r)
rho_moi_0 = MC_WB / (1 - MC_WB) * rho_bio_0
y0 = np.zeros(nc); y0[species.index("N2")] = 1.0

prop_gas    = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)
solid_cfg   = build_solid_prop_config(fuel_config)
gas_T_ref   = float(np.min(np.asarray(prop_gas["Tref"])))
MW_arr      = np.asarray(prop_gas["MW"])
trans_cfg   = build_transport_config(mode="constant", N=N, n_comp=nc, h_bed=80.0, h_wall=12.0)
bc_batch    = build_bc_config(n_comp=nc, P_out_bar=P_OUT, v_out=0.0)

solver_cfg = {
    "t_end":       T_END,
    "max_step":    20.0,
    "rtol":        1e-4,
    "atol":        1e-6,
    "method":      "BDF",
    "progress_bar": True,
    "t_eval":      None,
}

def make_params(tbc, bc=None):
    """Construye params completo a partir del thermal BC y del bc_config."""
    _bc = bc if bc is not None else bc_batch
    init = build_initial_c_config(
        P_init=P_OUT, Tg_init=300.0, Ts_init=300.0, y_init=y0,
        rho_biomass_init=rho_bio_0, rho_char_init=1e-6, rho_moisture_init=rho_moi_0,
        n_comp=nc, N=N, prop_gas=prop_gas, epsi_r=epsi_r, gas_T_ref=gas_T_ref,
    )
    return {
        "n_comp":nc, "N":N, "dz":dz, "Ai":Ai, "Di":Di, "Pi":Pi, "Po":Po,
        "prop_gas":prop_gas, "MW":MW_arr, "gas_T_ref":gas_T_ref,
        "bc_config":_bc, "trans_config":trans_cfg, "thermal_bc_config":tbc,
        "energy":True, "epsi_r":epsi_r, "dp0":dp0, "rho_char0":rho_char0,
        "fuel_config":fuel_config, "solid_config":solid_cfg, "species":species,
        "_cache":{},
    }, init["sv0"]

print(f"Parámetros comunes listos. N={N} (0D), L={L} m, T_end={T_END} s")
print(f"Biomasa inicial: {rho_bio_0:.1f} kg/m³_bed, humedad: {rho_moi_0:.1f} kg/m³_bed")

## 3A — Rampa de T_wall: calentamiento gradual desde 300 K hasta 1073 K

La pared se calienta a razón de **0.217 K/s** durante los primeros 3600 s, partiendo de temperatura
ambiente (300 K). La señal `ramp` satura en T_max=1073.15 K (800 °C).  
Se compara con el **caso constante** a T_wall=1073.15 K (referencia del test_01).  

**Pregunta:** ¿qué efecto tiene el calentamiento gradual sobre la pirólisis?
- El sólido tarda más en alcanzar la temperatura de activación cinética.
- La producción de gas se desplaza hacia tiempos mayores.
- La temperatura de gas y sólido al final deberían ser similares en ambos casos.

In [ ]:
# ── Señal: rampa de T_wall 300 → 1073.15 K en 3600 s ────────────────────────
T_wall_ramp = ramp(t_start=0.0, slope=(1073.15 - 300.0) / T_END,
                   value_init=300.0, value_max=1073.15)

# Verificar la señal antes de simular
t_check = [0, 600, 1200, 1800, 2400, 3000, 3600]
print("Verificación de T_wall_ramp(t):")
for tc in t_check:
    print(f"  t={tc:4d} s → T_wall = {resolve(T_wall_ramp, float(tc)):.1f} K")

# ── Caso 3A: T_wall rampa ────────────────────────────────────────────────────
tbc_3a = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_ramp,
    k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
params_3a, sv0_3a = make_params(tbc_3a)

print("\nSimulando 3A (T_wall rampa)...")
result_3a = run_step(params_3a, solver_cfg)
if result_3a.status != 0:
    print(f"⚠ Solver status={result_3a.status}: {result_3a.message}")

gasifier_3a = build_gasifier_results(result_3a, params_3a)

# ── Caso referencia: T_wall constante 1073.15 K ──────────────────────────────
params_ref, sv0_ref = make_params(
    build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=1073.15, k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
    )
)
print("Simulando caso referencia (T_wall=1073.15 K constante)...")
result_ref = run_step(params_ref, solver_cfg)
gasifier_ref = build_gasifier_results(result_ref, params_ref)

# ── Gráficas de comparación ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("3A — Rampa de T_wall vs. T_wall constante (0D, batch)", fontsize=12)

t_3a  = gasifier_3a._t_results
t_ref = gasifier_ref._t_results

# Panel 1: T_wall aplicada vs. tiempo
ax = axes[0, 0]
ax.plot(t_3a / 60, [resolve(T_wall_ramp, ti) for ti in t_3a], 'r-', label='rampa')
ax.axhline(1073.15, color='k', ls='--', label='constante')
ax.set_xlabel("t [min]"); ax.set_ylabel("T_wall [K]")
ax.set_title("Señal T_wall")
ax.legend(); ax.grid(True)

# Panel 2: Temperatura del sólido
ax = axes[0, 1]
ax.plot(t_3a / 60,  gasifier_3a._Ts_results[:, 0],  'r-',  label='rampa')
ax.plot(t_ref / 60, gasifier_ref._Ts_results[:, 0],  'k--', label='constante')
ax.set_xlabel("t [min]"); ax.set_ylabel("Ts [K]")
ax.set_title("Temperatura del sólido (c=0)")
ax.legend(); ax.grid(True)

# Panel 3: Biomasa restante
ax = axes[1, 0]
ax.plot(t_3a / 60,  gasifier_3a._rho_s_results[:, 0, 0],  'r-',  label='rampa')
ax.plot(t_ref / 60, gasifier_ref._rho_s_results[:, 0, 0],  'k--', label='constante')
ax.set_xlabel("t [min]"); ax.set_ylabel("ρ_biomasa [kg/m³_bed]")
ax.set_title("Biomasa restante")
ax.legend(); ax.grid(True)

# Panel 4: Presión del gas
ax = axes[1, 1]
ax.plot(t_3a / 60,  gasifier_3a._P_results[:, 0],  'r-',  label='rampa')
ax.plot(t_ref / 60, gasifier_ref._P_results[:, 0],  'k--', label='constante')
ax.set_xlabel("t [min]"); ax.set_ylabel("P [bar]")
ax.set_title("Presión del gas")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

print("\n--- Balances 3A (T_wall rampa) ---")
check_balances(gasifier_3a, params_3a)

## 3B — Escalón de T_wall: precalentamiento → temperatura de operación

La pared salta de **500 K a 1073 K** en t=600 s. Simula un arranque con precalentamiento previo
seguido de activación del horno a plena potencia.  

**Pregunta:** ¿cómo afecta el retraso de 600 s al inicio de la pirólisis?
El secado comienza ya en la primera fase (T_wall=500 K activa el drying);
la pirólisis se activa con fuerza al producirse el salto.

In [ ]:
T_wall_step = step(t_step=600.0, value_before=500.0, value_after=1073.15)

tbc_3b = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_step,
    k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
params_3b, sv0_3b = make_params(tbc_3b)

print("Simulando 3B (escalón de T_wall a t=600 s)...")
result_3b = run_step(params_3b, solver_cfg)
if result_3b.status != 0:
    print(f"⚠ Solver status={result_3b.status}: {result_3b.message}")
gasifier_3b = build_gasifier_results(result_3b, params_3b)

t_3b = gasifier_3b._t_results

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("3B — Escalón de T_wall: 500 K → 1073 K a t=600 s (0D, batch)", fontsize=12)

ax = axes[0]
ax.plot(t_3b / 60, [resolve(T_wall_step, ti) for ti in t_3b], 'b-', lw=2)
ax.set_xlabel("t [min]"); ax.set_ylabel("T_wall [K]")
ax.set_title("Señal T_wall"); ax.grid(True)

ax = axes[1]
ax.plot(t_3b / 60, gasifier_3b._Ts_results[:, 0],  'b-',  label='escalón')
ax.plot(t_ref / 60, gasifier_ref._Ts_results[:, 0], 'k--', label='cte 1073 K')
ax.set_xlabel("t [min]"); ax.set_ylabel("Ts [K]")
ax.set_title("Temperatura del sólido")
ax.legend(); ax.grid(True)

ax = axes[2]
ax.plot(t_3b / 60, gasifier_3b._rho_s_results[:, 0, 0],  'b-',  label='escalón')
ax.plot(t_ref / 60, gasifier_ref._rho_s_results[:, 0, 0], 'k--', label='cte 1073 K')
ax.set_xlabel("t [min]"); ax.set_ylabel("ρ_biomasa [kg/m³_bed]")
ax.set_title("Biomasa restante")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

print("\n--- Balances 3B (escalón T_wall) ---")
check_balances(gasifier_3b, params_3b)

## 3C — Pulso de Qwall: inyección periódica de calor

Se inyectan **500 W** al reactor durante 600 s, apagando el calentamiento el resto del tiempo.
T_wall queda en `adiabatic` — el calor entra SOLO durante el pulso.  

**Nota:** `heatfluxwall` con Qwall=callable(t) requiere que `wall_heat_flux` reciba el
valor resuelto via `_tbc = _resolve_cfg(thermal_bc_cfg, t, snap)`. Esto se hace
automáticamente en el RHS del gasificador.

In [ ]:
Q_pulse = pulse(t_start=0.0, t_end=600.0, value_on=500.0, value_off=0.0)

tbc_3c = build_thermal_bc_config(
    mode="heatfluxwall", Di=Di, Do=Do, e_wall=e_wall,
    Qwall=Q_pulse,
    k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
params_3c, sv0_3c = make_params(tbc_3c)

print("Simulando 3C (pulso Qwall: 500 W durante los primeros 600 s)...")
result_3c = run_step(params_3c, solver_cfg)
if result_3c.status != 0:
    print(f"⚠ Solver status={result_3c.status}: {result_3c.message}")
gasifier_3c = build_gasifier_results(result_3c, params_3c)

t_3c = gasifier_3c._t_results

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("3C — Pulso de Qwall: 500 W en t∈[0, 600 s] (0D, batch)", fontsize=12)

ax = axes[0]
ax.plot(t_3c / 60, [resolve(Q_pulse, ti) for ti in t_3c], 'g-', lw=2)
ax.set_xlabel("t [min]"); ax.set_ylabel("Qwall [W]")
ax.set_title("Señal Qwall"); ax.grid(True)

ax = axes[1]
ax.plot(t_3c / 60, gasifier_3c._Ts_results[:, 0], 'g-', label='pulso')
ax.set_xlabel("t [min]"); ax.set_ylabel("Ts [K]")
ax.set_title("Temperatura del sólido")
ax.legend(); ax.grid(True)

ax = axes[2]
ax.plot(t_3c / 60, gasifier_3c._P_results[:, 0], 'g-', label='pulso')
ax.set_xlabel("t [min]"); ax.set_ylabel("P [bar]")
ax.set_title("Presión del gas")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

print("\n--- Balances 3C (pulso Qwall) ---")
check_balances(gasifier_3c, params_3c)

## 3D — Piecewise: perfil de operación segmentado

Perfil de T_wall con 4 etapas:
- **0–600 s:** calentamiento lineal 300 → 700 K
- **600–1800 s:** plateau a 700 K (secado + inicio pirólisis)
- **1800–2400 s:** calentamiento final 700 → 1073 K
- **2400–3600 s:** plateau a 1073 K (pirólisis completa)  

Este tipo de perfil es representativo de un protocolo de calentamiento programado.

In [ ]:
T_wall_pw = piecewise(
    t_breakpoints=[0,    600,  1800,  2400,   3600],
    values=        [300,  700,   700,  1073.15, 1073.15],  # [K]
)

tbc_3d = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_pw,
    k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
params_3d, sv0_3d = make_params(tbc_3d)

print("Simulando 3D (perfil piecewise de T_wall)...")
result_3d = run_step(params_3d, solver_cfg)
if result_3d.status != 0:
    print(f"⚠ Solver status={result_3d.status}: {result_3d.message}")
gasifier_3d = build_gasifier_results(result_3d, params_3d)

t_3d = gasifier_3d._t_results

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("3D — Perfil piecewise de T_wall (0D, batch)", fontsize=12)

ax = axes[0]
ax.plot(t_3d / 60, [resolve(T_wall_pw, ti) for ti in t_3d], 'm-', lw=2)
ax.set_xlabel("t [min]"); ax.set_ylabel("T_wall [K]")
ax.set_title("Señal T_wall (piecewise)"); ax.grid(True)

ax = axes[1]
ax.plot(t_3d / 60, gasifier_3d._Ts_results[:, 0], 'm-',  label='piecewise')
ax.plot(t_ref / 60, gasifier_ref._Ts_results[:, 0], 'k--', label='cte 1073 K')
ax.set_xlabel("t [min]"); ax.set_ylabel("Ts [K]")
ax.set_title("Temperatura del sólido")
ax.legend(); ax.grid(True)

ax = axes[2]
ax.plot(t_3d / 60, gasifier_3d._rho_s_results[:, 0, 0], 'm-',  label='piecewise')
ax.plot(t_ref / 60, gasifier_ref._rho_s_results[:, 0, 0], 'k--', label='cte 1073 K')
ax.set_xlabel("t [min]"); ax.set_ylabel("ρ_biomasa [kg/m³_bed]")
ax.set_title("Biomasa restante")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

print("\n--- Balances 3D (piecewise T_wall) ---")
check_balances(gasifier_3d, params_3d)

## 3E — Control proporcional: T_wall ajustada para mantener Ts_mean en 700 K

Un controlador P ajusta T_wall en cada paso del RHS para mantener la temperatura media del
sólido (`Ts_mean` del snap) en el setpoint de **700 K**.  
El snap se construye en el RHS paso 2 y se pasa al evaluador de thermal BC.  

**Parámetros del controlador:**
- `setpoint = 700 K`
- `gain = 3.0 K_wall/K_error`
- `output_bias = 800 K` (T_wall nominal)
- `output_min = 300 K`, `output_max = 1200 K`

**Qué observar:**
- T_wall se mueve para contrarrestar la desviación de Ts respecto al SP.
- El controlador P no elimina el error en estado estacionario (no hay integral).
- Los balances deben cerrar igual que con T_wall constante.

In [ ]:
SP_Ts = 700.0   # [K] setpoint de temperatura del sólido

ctrl_P = proportional(
    setpoint=SP_Ts,
    gain=3.0,
    channel_in=lambda snap: snap.get("Ts_mean", 300.0),
    output_bias=800.0,
    output_min=300.0,
    output_max=1200.0,
)

tbc_3e = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=ctrl_P,            # callable(t, snap) → float
    k_wall=16.5, rho_wall=7950.0, Cp_wall=510.0,
)
params_3e, sv0_3e = make_params(tbc_3e)

print(f"Simulando 3E (control P de T_wall, SP_Ts={SP_Ts} K)...")
result_3e = run_step(params_3e, solver_cfg)
if result_3e.status != 0:
    print(f"⚠ Solver status={result_3e.status}: {result_3e.message}")
gasifier_3e = build_gasifier_results(result_3e, params_3e)

t_3e = gasifier_3e._t_results

# Reconstruir T_wall efectiva desde el snap (post-proceso)
Ts_hist_3e = gasifier_3e._Ts_results[:, 0]
T_wall_eff = np.array([
    float(SP_Ts + 3.0 * (SP_Ts - Ts_hist_3e[i]) + 800.0 - SP_Ts)
    for i in range(len(t_3e))
])
T_wall_eff = np.clip(T_wall_eff, 300.0, 1200.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f"3E — Control P: T_wall ajustada para Ts_mean = {SP_Ts} K (0D, batch)", fontsize=12)

ax = axes[0]
ax.plot(t_3e / 60, T_wall_eff, 'darkorange', lw=2)
ax.axhline(800.0, color='k', ls='--', alpha=0.5, label='output_bias')
ax.set_xlabel("t [min]"); ax.set_ylabel("T_wall efectiva [K]")
ax.set_title("T_wall del controlador P")
ax.legend(); ax.grid(True)

ax = axes[1]
ax.plot(t_3e / 60, Ts_hist_3e, 'darkorange', label='control P')
ax.axhline(SP_Ts, color='r', ls='--', label=f'SP={SP_Ts} K')
ax.set_xlabel("t [min]"); ax.set_ylabel("Ts [K]")
ax.set_title("Temperatura del sólido")
ax.legend(); ax.grid(True)

ax = axes[2]
ax.plot(t_3e / 60, gasifier_3e._rho_s_results[:, 0, 0], 'darkorange', label='control P')
ax.plot(t_ref / 60, gasifier_ref._rho_s_results[:, 0, 0], 'k--', label='cte 1073 K')
ax.set_xlabel("t [min]"); ax.set_ylabel("ρ_biomasa [kg/m³_bed]")
ax.set_title("Biomasa restante")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

print("\n--- Balances 3E (control P T_wall) ---")
check_balances(gasifier_3e, params_3e)

## Conclusiones

**Qué demuestra este test:**
- Los tres tipos de señal BC (constante, callable(t), callable(t, snap)) funcionan
  correctamente con el gasificador 0D — los balances cierran en todos los casos.
- El solver BDF maneja sin problemas discontinuidades suaves (escalón, pulso); solo los
  escalones muy abruptos pueden requerir reducir `max_step` alrededor del punto de cambio.
- El snap del RHS (paso 2) se construye correctamente: los controladores reciben
  `Ts_mean` actualizado en cada paso de integración.

**Diferencias clave entre las señales:**

| Caso | T_wall aplicada | Efecto sobre pirólisis |
|------|----------------|------------------------|
| Ref  | 1073 K constante | Conversión rápida desde t=0 |
| 3A (ramp) | 300 → 1073 K en 3600 s | Conversión más lenta; temperatura final similar |
| 3B (step) | 500 K → 1073 K a t=10 min | Secado activo en fase 1; pirólisis se activa con el salto |
| 3C (pulse) | 500 W durante 10 min | Conversión parcial; sólido no alcanza temperatura de pirólisis |
| 3D (piecewise) | 4 etapas programadas | Control de la velocidad de conversión por tramos |
| 3E (ctrl P) | Ajustada por Ts_mean | Ts regulada cerca del SP; no elimina error en estado estacionario |

**Qué probar a continuación:**
- Añadir el modo `onoff` en el controlador de T_wall para comparar con el controlador P.
- Probar señales en `v_gas_in` para simular arranque/parada de la inyección de agente.
- Subir a 1D (N=10) para ver la respuesta espacial a señales temporales.
- Ver `test_gasifier_04_0D_control.ipynb` para técnicas de optimización y control avanzado.